<a href="https://colab.research.google.com/github/hamshini1413/hamshini_gen_ai_foundation/blob/main/14_Custom_CUDA_Kernel_Integration_for_Accelerated_Activation_Functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip -q install ninja

In [31]:
import torch
from torch.utils.cpp_extension import load_inline
import time

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4


In [32]:
cpp_source = r'''
#include <torch/extension.h>

torch::Tensor swiglu_cuda(torch::Tensor x, torch::Tensor gate);

torch::Tensor swiglu(torch::Tensor x, torch::Tensor gate)
{
    return swiglu_cuda(x, gate);
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m)
{
    m.def("swiglu", &swiglu, "SwiGLU CUDA");
}
'''

In [33]:
cuda_source = r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

template <typename scalar_t>
__global__ void swiglu_kernel(
    const scalar_t* x,
    const scalar_t* gate,
    scalar_t* output,
    int size)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if(idx < size)
    {
        scalar_t g = gate[idx];

        scalar_t sigmoid =
            1.0 / (1.0 + exp(-g));

        scalar_t swish = g * sigmoid;

        output[idx] = x[idx] * swish;
    }
}

torch::Tensor swiglu_cuda(torch::Tensor x,
                          torch::Tensor gate)
{
    auto output = torch::zeros_like(x);

    int size = x.numel();

    const int threads = 256;
    const int blocks = (size + threads - 1) / threads;

    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(),
    "swiglu_cuda",
    ([&]
    {
        swiglu_kernel<scalar_t><<<blocks,threads>>>(
            x.data_ptr<scalar_t>(),
            gate.data_ptr<scalar_t>(),
            output.data_ptr<scalar_t>(),
            size);
    }));

    return output;
}
'''

In [34]:
swiglu_ext = load_inline(
    name="swiglu_extension",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=None,
    verbose=False,
)

In [35]:
device = "cuda"

x = torch.randn(1000000, device=device)
gate = torch.randn(1000000, device=device)

output = swiglu_ext.swiglu(x, gate)

print(output[:10])

tensor([ 4.1581e-01, -2.1629e-02,  2.7727e-01, -4.1989e-01,  2.2197e-01,
        -3.8361e+00, -5.8834e-02, -2.7368e-01,  3.6587e-03,  4.7877e-01],
       device='cuda:0')


In [36]:
def pytorch_swiglu(x, gate):
    return x * (gate * torch.sigmoid(gate))

In [37]:
torch.cuda.synchronize()

start = time.time()

for _ in range(100):
    y1 = pytorch_swiglu(x, gate)

torch.cuda.synchronize()

torch_time = time.time() - start


torch.cuda.synchronize()

start = time.time()

for _ in range(100):
    y2 = swiglu_ext.swiglu(x, gate)

torch.cuda.synchronize()

cuda_time = time.time() - start


print(f"PyTorch Time : {torch_time:.4f} sec")
print(f"CUDA Time    : {cuda_time:.4f} sec")
print(f"Speedup      : {torch_time/cuda_time:.2f}x")

PyTorch Time : 0.0145 sec
CUDA Time    : 0.0220 sec
Speedup      : 0.66x


In [38]:
max_error = torch.max(torch.abs(y1 - y2))

print("Maximum Error:", max_error.item())

Maximum Error: 9.5367431640625e-07
